# DS behavior video conversion jobs

Discover `Flir*.avi` files inside `*-good` folders and submit one SLURM job per AVI to convert it to MP4 on the cluster.

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'miscellaneous' else NOTEBOOK_DIR

for path in (REPO_ROOT, REPO_ROOT / 'miscellaneous'):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from ds_behav_cluster_jobs import (
    DEFAULT_CLUSTER_HOST,
    DEFAULT_CONDA_ENV,
    DEFAULT_LOG_DIR,
    DEFAULT_OUTPUT_DIR,
    DEFAULT_PIPELINE_WORKDIR,
    DEFAULT_RUNNER_SCRIPT,
    DEFAULT_USERNAME,
    connect_ssh,
    discover_flir_avi_jobs,
    print_job_summary,
    submit_conversion_jobs,
    wait_for_jobs,
)


In [ ]:
CKII_data_folders = [
    '/Volumes/adam-lab/Adam-Lab-Shared/Data/renana_malka/pAce21/2025-08-06_pAce21_PR/Awake',
    '/Volumes/adam-lab/Adam-Lab-Shared/Data/renana_malka/pAce38/2025-11-26_pAce38/PX/post/Awake',
    '/Volumes/adam-lab/Adam-Lab-Shared/Data/renana_malka/pAce45/2026-01-18_pAce45/PX/post/Awake',
    '/Volumes/adam-lab/Adam-Lab-Shared/Data/renana_malka/pAce46/2026-02-22_pAce46/PR/post/Awake',
    '/Volumes/adam-lab/Adam-Lab-Shared/Data/renana_malka/pAce47/2026-01-28_pAce47/PX/post/Awake',
    '/Volumes/adam-lab/Adam-Lab-Shared/Data/renana_malka/pAce50/2026-03-17_pAce50_PRL/Awake',
]

CLUSTER_HOST = DEFAULT_CLUSTER_HOST
USERNAME = DEFAULT_USERNAME
CONDA_ENV = 'caiman'
PIPELINE_WORKDIR = DEFAULT_PIPELINE_WORKDIR
RUNNER_SCRIPT = DEFAULT_RUNNER_SCRIPT
OUTPUT_DIR = DEFAULT_OUTPUT_DIR
LOG_DIR = DEFAULT_LOG_DIR
FFMPEG_BINARY = 'ffmpeg'

jobs = discover_flir_avi_jobs(CKII_data_folders, output_dir=OUTPUT_DIR)
print_job_summary(jobs)
print(f'\nConda env on cluster: {CONDA_ENV}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'Log dir: {LOG_DIR}')


In [ ]:
ssh = connect_ssh(cluster_host=CLUSTER_HOST, username=USERNAME)
dry_run_results = submit_conversion_jobs(
    ssh,
    jobs,
    pipeline_workdir=PIPELINE_WORKDIR,
    conda_env=CONDA_ENV,
    runner_script=RUNNER_SCRIPT,
    log_dir=LOG_DIR,
    ffmpeg_binary=FFMPEG_BINARY,
    cpus=2,
    mem_gb=32,
    time_limit='08:00:00',
    crf=17,
    preset='slow',
    overwrite=False,
    dry_run=True,
)

for item in dry_run_results:
    print(item['command'])
    print()


In [ ]:
# Run after verifying the dry-run commands above.
submit_results = submit_conversion_jobs(
    ssh,
    jobs,
    pipeline_workdir=PIPELINE_WORKDIR,
    conda_env=CONDA_ENV,
    runner_script=RUNNER_SCRIPT,
    log_dir=LOG_DIR,
    ffmpeg_binary=FFMPEG_BINARY,
    cpus=2,
    mem_gb=32,
    time_limit='08:00:00',
    crf=17,
    preset='slow',
    overwrite=False,
    dry_run=False,
)

for item in submit_results:
    print(item['output'] or item['error'])

job_ids = [item['job_id'] for item in submit_results if item['job_id']]
wait_for_jobs(ssh, job_ids, poll_interval=60)


In [ ]:
from ds_behav_cluster_jobs import cluster_to_local_path

print(f"Log dir (local): {cluster_to_local_path(LOG_DIR)}")
